In [1]:
essay = """India in the Age of AI
As the world enters a transformative era defined by artificial intelligence (AI), India stands at a critical juncture — one where it can either emerge as a global leader in AI innovation or risk falling behind in the technology race. The age of AI brings with it immense promise as well as unprecedented challenges, and how India navigates this landscape will shape its socio-economic and geopolitical future.

India's strengths in the AI domain are rooted in its vast pool of skilled engineers, a thriving IT industry, and a growing startup ecosystem. With over 5 million STEM graduates annually and a burgeoning base of AI researchers, India possesses the intellectual capital required to build cutting-edge AI systems. Institutions like IITs, IIITs, and IISc have begun fostering AI research, while private players such as TCS, Infosys, and Wipro are integrating AI into their global services. In 2020, the government launched the National AI Strategy (AI for All) with a focus on inclusive growth, aiming to leverage AI in healthcare, agriculture, education, and smart mobility.

One of the most promising applications of AI in India lies in agriculture, where predictive analytics can guide farmers on optimal sowing times, weather forecasts, and pest control. In healthcare, AI-powered diagnostics can help address India’s doctor-patient ratio crisis, particularly in rural areas. Educational platforms are increasingly using AI to personalize learning paths, while smart governance tools are helping improve public service delivery and fraud detection.

However, the path to AI-led growth is riddled with challenges. Chief among them is the digital divide. While metropolitan cities may embrace AI-driven solutions, rural India continues to struggle with basic internet access and digital literacy. The risk of job displacement due to automation also looms large, especially for low-skilled workers. Without effective skilling and re-skilling programs, AI could exacerbate existing socio-economic inequalities.

Another pressing concern is data privacy and ethics. As AI systems rely heavily on vast datasets, ensuring that personal data is used transparently and responsibly becomes vital. India is still shaping its data protection laws, and in the absence of a strong regulatory framework, AI systems may risk misuse or bias.

To harness AI responsibly, India must adopt a multi-stakeholder approach involving the government, academia, industry, and civil society. Policies should promote open datasets, encourage responsible innovation, and ensure ethical AI practices. There is also a need for international collaboration, particularly with countries leading in AI research, to gain strategic advantage and ensure interoperability in global systems.

India’s demographic dividend, when paired with responsible AI adoption, can unlock massive economic growth, improve governance, and uplift marginalized communities. But this vision will only materialize if AI is seen not merely as a tool for automation, but as an enabler of human-centered development.

In conclusion, India in the age of AI is a story in the making — one of opportunity, responsibility, and transformation. The decisions we make today will not just determine India’s AI trajectory, but also its future as an inclusive, equitable, and innovation-driven society."""

In [2]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from typing import TypedDict, Annotated
from dotenv import load_dotenv
import operator

In [3]:
load_dotenv()
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=1.0)

In [4]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback for essay")
    score: int = Field(description="score out of 10", ge = 0, le=10)

In [5]:
structured_model = model.with_structured_output(EvaluationSchema)

In [6]:
class UPSCstate(TypedDict):
    essay: str
    COT_feedback: str
    DOA_feedback: str
    Language_feedback: str
    individual_score: Annotated[list[int],operator.add]
    final_feedback: str
    avg_score: float

In [7]:
def evaluate_thought(state: UPSCstate):
    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    res = structured_model.invoke(prompt)
    return {"COT_feedback": res.feedback, "individual_score": [res.score]}


In [8]:
def evaluate_language(state: UPSCstate):
    prompt = f'Evaluate the Language quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    res = structured_model.invoke(prompt)
    return {"Language_feedback": res.feedback, "individual_score": [res.score]}

In [9]:
def evaluate_Analysis(state: UPSCstate):
    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    res = structured_model.invoke(prompt)
    return {"DOA_feedback": res.feedback, "individual_score": [res.score]}

In [10]:
def final_evaluation(state: UPSCstate):
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["Language_feedback"]} \n depth of analysis feedback - {state["DOA_feedback"]} \n clarity of thought feedback - {state["COT_feedback"]}'
    res = model.invoke(prompt).content

    score = sum(state["individual_score"])/ len(state["individual_score"])
    return {"final_feedback": res, "avg_score": score}

    

In [11]:
graph = StateGraph(UPSCstate)

graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_Analysis',evaluate_Analysis)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'evaluate_thought')
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_Analysis')
graph.add_edge('evaluate_Analysis', 'final_evaluation')
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')
graph.add_edge('final_evaluation',END)

workflow = graph.compile()


In [12]:
initial_state = {'essay': essay}
final_state = workflow.invoke(initial_state)
print(final_state)

{'essay': "India in the Age of AI\nAs the world enters a transformative era defined by artificial intelligence (AI), India stands at a critical juncture — one where it can either emerge as a global leader in AI innovation or risk falling behind in the technology race. The age of AI brings with it immense promise as well as unprecedented challenges, and how India navigates this landscape will shape its socio-economic and geopolitical future.\n\nIndia's strengths in the AI domain are rooted in its vast pool of skilled engineers, a thriving IT industry, and a growing startup ecosystem. With over 5 million STEM graduates annually and a burgeoning base of AI researchers, India possesses the intellectual capital required to build cutting-edge AI systems. Institutions like IITs, IIITs, and IISc have begun fostering AI research, while private players such as TCS, Infosys, and Wipro are integrating AI into their global services. In 2020, the government launched the National AI Strategy (AI for 

In [13]:
print(final_state['final_feedback'])

Here's a summarized feedback based on the provided reviews:

**Overall, the essay offers a well-structured and comprehensive overview of India's position in the Age of AI. The author effectively balances the numerous opportunities with significant challenges, highlighting India's strengths in its skilled workforce and IT industry, while also addressing critical hurdles like the digital divide, job displacement, and data privacy concerns. The essay's clear language and logical flow make it an engaging read, and the call for a multi-stakeholder, human-centered approach is a strong concluding point.**

**To further enhance the essay, consider incorporating more specific examples and statistics to substantiate points, providing a more detailed discussion on the nuances of the digital divide, and elaborating on the ethical considerations and potential economic impacts of AI in India.**


In [15]:
print(final_state['individual_score'])

[7, 9, 9]
